# 02 · Synthetic criteria pool (label-free selection and the ranking of the methods)

**Purpose.** Without target labels, the second phase of every method needs a label-free rule to pick one of its
searched configurations. This notebook builds the *shared pool* on which the paper compares those rules and the
methods: for each of the 24 synthetic units (class effect $d_C \in \{0.6, 1, 1.5, 2.5\}$ x shift
$\{0.5, 1.5, 3\}$ x two rotations), each of the eight methods (MIL-CREDA and seven comparators) trains the same 100
seeded random phase-2 configurations for 100 epochs, starting from the unit's frozen phase-1 base. Every trial
records its balanced accuracy (BA) at the base's cut on 120 fresh simulated target subjects (the *oracle*, whose
labels only the simulator knows) and the outputs from which each label-free criterion is computed offline.
Each criterion then picks, per (unit, method), the trial with the highest score (first maximum in trial order).

**Paper items produced here**

* Table `tab:selection-dev`: BA of the configuration each criterion picks (rows) for each method (columns), the
  oracle, the method ranks under cef, and the Holm-corrected Wilcoxon tests with Cliff's delta.
* Figure `fig:synthetic-map` (`synthetic-map-methods.pdf`): BA of the cef pick minus the first phase per
  (class effect, shift) cell, for MIL-CREDA and the comparators' mean.
* The numbers of the two paragraphs around them (criteria ranking, method ranking, map).
* `milcreda_picks.json` and `comparator_picks.json`: the cef picks (trial and settings) that **stage 03**
  (synthetic mechanisms) and **stage 05** (attention influence) retrain and analyse. They are part of this stage's
  stored summary.

**Where the data comes from.** Everything is simulated (`MIL_CREDA_EEG.eeg_sim` through `xpilot.build_sides`);
no external dataset is read. The frozen phase-1 bases are the protocol v82 final-rule bases, stored with this stage
in `Results/paper/02_synthetic_criteria_pool/inputs/bases/` (stage 01 regenerates them).

In [ ]:
# Setup: make `src/` importable, locally (notebook inside the repository) or on Kaggle (clone the repository).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/Domain_Adaptation"
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / "src" / "MIL_CREDA_EEG").is_dir()), None)
if root is None:                       # Kaggle / Colab: clone the public repository next to the notebook
    root = Path("/kaggle/working/Domain_Adaptation") if Path("/kaggle/working").exists() else here / "Domain_Adaptation"
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    # the simulator needs MNE and NeuroDSP (versions of the campaign's run-config.json)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mne==1.10.1", "neurodsp==2.2.1"], check=True)
sys.path.insert(0, str(root / "src"))
print("repository:", root)

## Parameters

The module `MIL_CREDA_EEG.reproduce.stage02_criteria` calls the entrypoint the paper used,
`MIL_CREDA_EEG.criteria_pool.run`, with the campaign's parameters. The campaign ran as Kaggle family `sx`: one
notebook per (unit, trial shard), 24 x 8 = **192 CPU notebooks**, each with the `run-config.json` below.

| parameter | value | source |
|---|---|---|
| units | 24: `d{0p6,1,1p5,2p5}-s{0p5,1p5,3}-r{0,1}` | `run-config.json` `unit_key` of the 192 `sx-*` jobs |
| phase-2 seed / shards / processes / threads | 1 / 8 / 4 / 1 | `run-config.json` `run.kwargs` |
| trials per method, epochs | 100, 100 | `criteria_pool.TRIALS`, `criteria_pool.EPOCHS2` |
| configurations | seeded `RandomSampler` (seed 20261115 + 1000 k + method index) over each method's full space | `criteria_pool.draw` |
| learning rate | annealed $\eta_2(1+10p)^{-0.75}$ | `criteria_pool.train`, `lr_schedule` |
| evaluation pool | 120 fresh target subjects, BA at the base's cut | `oracle_screen.EVAL_SUBJECTS`, `oracle_screen.eval_pool` |
| phase-1 bases | v82 final-rule bases, seed 1 (tag `archive/ieee-pool-slim`, commit 820ce461) | `inputs/bases/` |
| criteria | cef, BNM, IM, ClassAMI, entropy, SND, source BA, centroid BA | `crit3.py` (ported) |
| tests | Friedman; Wilcoxon signed-rank by unit, monotone Holm; Cliff's delta; Nemenyi CD (q = 3.102) | `h3_pool.py`, `h1_pool24.py` (ported) |

The code at commit 820ce461 that the campaign ran is byte-identical, for `criteria_pool`, `oracle_screen`,
`xcompare` and `xpilot`, to this repository's.

**Cost of the full run.** CPU only, no GPU. The campaign records of the `sx` jobs did not keep their run times, so the cost is a measured estimate: on an Apple M2 core (1 torch thread, unit `d1p5-s1p5-r0`), one 20-epoch trial took 11.9 s for MIL-CREDA and 3.0 s for COT, i.e. about 60 s and 15 s at 100 epochs. With `criteria_pool.SLOWNESS` (CASUAL 6.0, MIL-CREDA 3.6, CLUDA 1.5, UDA-DDA 1.4, the other four 1.0 relative to the fastest), one trial index of all eight methods costs about 4 min of one core, so the full pool (24 units x 100 trials) is about 160 core-hours. One `sx` notebook (one unit, 12-13 trials of each method, 4 processes on a 4-core Kaggle CPU) therefore needs roughly 15-30 min plus about 1 min of setup per process; all 192 in one account would take days, which is why the paper spread them over several Kaggle accounts. The smoke run (`REPRO_SMOKE=1`) takes about 1 min.

**Splitting.** `SPLIT = None` runs the 192 items in this session; `SPLIT = (k, n)` runs items k, k+n, k+2n, ... of
the list, so n accounts (or n sessions) each run `(0, n)`, `(1, n)`, ..., `(n-1, n)`. The paper used 192 notebooks,
i.e. `n = 192`, spread over several Kaggle accounts.

In [ ]:
import json
from MIL_CREDA_EEG import repro
from MIL_CREDA_EEG.reproduce import stage02_criteria as S

SPLIT = None  # or (k, n) to run portion k of n in one Kaggle account
SMOKE = os.environ.get("REPRO_SMOKE") == "1"   # one MIL-CREDA trial of 2 epochs, only to prove the wiring

STAGE_DIR = repro.results_dir(S.STAGE)
if SMOKE:                               # never mix a smoke result with the stored summary
    import tempfile
    WORK = Path(tempfile.mkdtemp(prefix="stage02-smoke-"))
    RAW, SUMMARY = WORK / "raw", WORK / "summary"
else:
    RAW, SUMMARY = STAGE_DIR / "raw", STAGE_DIR / "summary"
items = repro.portion(S.UNITS, SPLIT)
print(f"{len(items)} of {len(S.UNITS)} (unit, shard) items in this portion, e.g. {items[:3]}")
print("bases:", S.bases_dir())

## Run

Each item trains the 100/8 trials of its shard for all eight methods and writes
`raw/sx-<unit>-<shard>/criteria-<unit>-k1-sh<shard>of8.json` plus one `npz` of outputs per trial. Before training,
`run` checks that every base is the v82 base the paper used (its cut must equal the stored one). Raw outputs are
large (about 18 500 `npz` files for the full pool) and are not committed.

In [ ]:
files = S.run(split=SPLIT, smoke=SMOKE, out=RAW)
print(len(files), "result file(s); first:", files[:1])

## Merge

When the work was split, copy every account's `raw/` folder into this stage's `Results/paper/02_synthetic_criteria_pool/raw/`
(any sub-folder layout works: every `criteria-*.json` below it is read). `merge` then

1. computes the eight label-free criteria of every finite trial (port of `crit3.py` + `export_trials.py`) and
   writes `criteria_pool_trials.json.gz`;
2. writes the cef picks for stages 03 and 05 (port of `mkpicks82.py`);
3. when all 24 units x 8 methods are present, derives `selection.json` (criteria, `h3_pool.py`), `methods.json`
   (methods, `h1_pool24.py`) and `map_cells.json` (figure data).

Without raw outputs it keeps the stored summary, which holds the paper campaign's own files (the raw `sx` outputs of
the campaign were not kept). The phase-1 BA of each base (`phase1_bases.json`) is stored; `rebuild_phase1=True`
recomputes it (port of `eval_bases.py`, about one minute per unit, no training).

In [ ]:
summary = S.merge(raw=RAW, out=SUMMARY)
print(sorted(p.name for p in Path(summary).iterdir()))

## Render

The table and the figure are drawn from the stored summary only (no training): `tables/selection-dev.{tex,csv}`
and `figures/synthetic-map-methods.{pdf,png}` (port of the paper's `make_map.py`).

In [ ]:
import pandas as pd
from IPython.display import Image, display

out = S.render()
display(pd.read_csv(STAGE_DIR / "tables" / "selection-dev.csv", dtype=str, keep_default_na=False))
display(Image(filename=str(STAGE_DIR / "figures" / "synthetic-map-methods.png")))
print(open(out["table"]).read()[:400], "...")

## Comparison with the paper

Each row is a number the paper states (`main.tex` line in the name) next to the value rendered from the stored
summary, rounded to the paper's printed precision; the tolerance is 0 because no training is involved. A full
re-run would draw the same configurations and seeds, but CPU thread scheduling and library versions can move a
trial's BA slightly, so a re-run's numbers may differ in the last digit.

In [ ]:
rows = repro.compare(S.paper_checks())
print("all within tolerance:", all(r["within"] for r in rows))